In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
import os

DATASET_PATH = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", NUM_CLASSES)

2026-03-21 13:00:07.537503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774098007.932565      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774098008.042639      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774098009.008896      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774098009.008931      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774098009.008934      55 computation_placer.cc:177] computation placer alr

Found 54305 files belonging to 38 classes.
Using 43444 files for training.


I0000 00:00:1774098090.011349      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774098090.017323      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 54305 files belonging to 38 classes.
Using 10861 files for validation.
Classes: 38


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # freeze base model

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.resnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 15

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15


I0000 00:00:1774098276.198191     135 cuda_dnn.cc:529] Loaded cuDNN version 91002


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 233s 163ms/step - accuracy: 0.7920 - loss: 0.7848 - val_accuracy: 0.9387 - val_loss: 0.1993
Epoch 2/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 220s 162ms/step - accuracy: 0.9275 - loss: 0.2526 - val_accuracy: 0.9471 - val_loss: 0.1838
Epoch 3/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 221s 163ms/step - accuracy: 0.9365 - loss: 0.2260 - val_accuracy: 0.9510 - val_loss: 0.1791
Epoch 4/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 222s 163ms/step - accuracy: 0.9471 - loss: 0.1791 - val_accuracy: 0.9570 - val_loss: 0.1515
Epoch 5/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 221s 163ms/step - accuracy: 0.9490 - loss: 0.1703 - val_accuracy: 0.9561 - val_loss: 0.1760
Epoch 6/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 221s 163ms/step - accuracy: 0.9504 - loss: 0.1763 - val_accuracy: 0.9551 - val_loss: 0.1736
Epoch 7/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 222s 163ms/step - accuracy: 0.9542 - loss: 0.1542 - val_accuracy: 0.9556 - val_loss: 0.1810
Epoch 8/15
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 221s 162ms/step - accuracy: 0.9

In [4]:
loss, acc = model.evaluate(val_ds)
print("Validation Loss:", loss)
print("Validation Accuracy:", acc)

340/340 ━━━━━━━━━━━━━━━━━━━━ 44s 130ms/step - accuracy: 0.9712 - loss: 0.1366
Validation Loss: 0.12417242676019669
Validation Accuracy: 0.9717337489128113
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 1

KeyboardInterrupt: 

In [5]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=class_names))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.98      0.98      0.98        45
                                 Apple___Black_rot       1.00      1.00      1.00        46
                          Apple___Cedar_apple_rust       1.00      0.95      0.97        20
                                   Apple___healthy       0.97      0.99      0.98       112
                               Blueberry___healthy       0.99      1.00      1.00       107
          Cherry_(including_sour)___Powdery_mildew       1.00      0.97      0.99        71
                 Cherry_(including_sour)___healthy       1.00      0.96      0.98        48
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.90      0.90      0.90        31
                       Corn_(maize)___Common_rust_       1.00      1.00      1.00        76
               Corn_(maize)___Northern_Leaf_Blight       0.96      0.96      0.

In [ ]:
model.save("/kaggle/working/plant_disease_resnet.h5")